# SingBERT Distillation — NS Commitment Buyin Axis (Stage 5b — v1, Single-Head)

Distils gpt-4.1-mini commitment labels onto `zanelim/singbert-large-sg` for the **buyin axis only**.
Single-head architecture — one encoder, one 3-class head.

- **Axis:** buyin — committed / uncommitted / neutral (personal investment in own service)

## Design (lessons from Stage 5a and dual-head v1)
- **Split BEFORE any replication** — replicate-then-split leaks duplicates into val
- **Human rows → evaluation ONLY, LLM rows → train ONLY** — mixing gave fake val kappa=1.0
- **No class weights** — use row replication instead (committed × 10, uncommitted × 10, neutral × 1)
- Replicate AFTER splitting, never before
- LR=2e-5, 4 epochs max, best model by val kappa
- `save_total_limit=1` — BERT-large checkpoints are ~1.3 GB; Kaggle disk is 20 GB

## Data
- **Train:** `commitment_llm_enrich_queue.csv` — `llm_buyin` column
- **Test (held-out):** `commitment_testset_queue.csv` — `human_label` column for buyin

## Gate
- Test kappa ≥ 0.45 AND uncommitted recall ≥ 0.40 AND committed recall ≥ 0.30

## Kaggle setup
- Attach dataset containing `commitment_llm_enrich_queue.csv` + `commitment_testset_queue.csv`
- GPU: T4 (BERT-large fits batch 16 at max_len 256). Expected runtime ~30–45 min for 4 epochs.

In [ ]:
# Upgrade transformers + peft together to avoid EncoderDecoderCache import mismatch.
# Kaggle's pre-installed peft requires transformers>=4.43.0; pinning breaks it. NO version pins.
!pip install -q -U transformers peft datasets scikit-learn

In [ ]:
import glob
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Auto-discover input files (works regardless of dataset slug name) ──────
def find_input_file(*filenames: str) -> Path:
    """Return the first match for any of the candidate filenames.

    Searches /kaggle/input recursively, then the local working dir
    (useful for local testing).
    """
    for filename in filenames:
        matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
        if matches:
            return Path(matches[0])
        local = Path(filename)
        if local.exists():
            return local
    raise FileNotFoundError(
        f"None of {filenames} found under /kaggle/input/ or cwd.\n"
        f"Attach the dataset containing the labelled enrich CSV via Add Data."
    )

TRAIN_PATH = find_input_file("commitment_llm_enrich_queue.csv", "commitment_llm_enrich_labelled.csv")
TEST_PATH  = find_input_file("commitment_testset_queue.csv")
print(f"Train (LLM labels) : {TRAIN_PATH}")
print(f"Test  (human)      : {TEST_PATH}")

OUTPUT_DIR = Path("/kaggle/working/singbert_ns_buyin")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ───────────────────────────────────────────────────────────
MODEL_NAME = "zanelim/singbert-large-sg"
MAX_LEN    = 256
BATCH_SIZE = 16      # BERT-large fits 16 on T4 16GB
GRAD_ACCUM = 2       # effective batch = 32
LR         = 2e-5
EPOCHS     = 4
VAL_FRAC   = 0.10

# ── Buyin label encoding ──────────────────────────────────────────────────
LABEL2ID = {"committed": 0, "uncommitted": 1, "neutral": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3
LABEL_ORDER = ["committed", "uncommitted", "neutral"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load LLM-labelled enrich data (training pool) ──────────────────────────
raw = pd.read_csv(TRAIN_PATH)
print(f"Raw enrich rows: {len(raw):,}")

# Schema guard
required_cols = {"chunk_id", "text", "llm_buyin"}
missing = required_cols - set(raw.columns)
assert not missing, (
    f"Enrich CSV missing columns {missing}. Got: {list(raw.columns)}."
)

# Clean labels
raw["llm_buyin"] = raw["llm_buyin"].astype(str).str.strip().str.lower()
raw = raw.dropna(subset=["text"])
raw["text"] = raw["text"].astype(str).str.strip()
raw = raw[raw["text"].str.len() > 0]

# Drop rows with invalid labels
valid_mask = raw["llm_buyin"].isin(LABEL2ID)
bad_labels = sorted(set(raw["llm_buyin"]) - set(LABEL2ID) - {"", "nan", "none"})
if bad_labels:
    print(f"WARNING — dropping rows with unexpected buyin labels: {bad_labels}")

llm_df = raw[valid_mask].drop_duplicates("chunk_id").reset_index(drop=True)
n_dropped = len(raw) - len(llm_df)

assert len(llm_df) >= 1000, (
    f"Only {len(llm_df)} labelled rows found — has --annotate-enrich been run? "
    f"({n_dropped} rows dropped due to empty/invalid buyin labels)"
)
print(f"Labelled rows: {len(llm_df):,}  ({n_dropped:,} dropped)")

print(f"\nBuyin label distribution:")
dist = llm_df["llm_buyin"].value_counts()
print(dist)

# ── WARNING: committed < 5% indicates FAISS enrichment needed ─────────────
committed_pct = (llm_df["llm_buyin"] == "committed").mean()
if committed_pct < 0.05:
    print(f"\nWARNING: committed rows are only {committed_pct:.1%} of the pool."
          " Recommend running FAISS enrichment before proceeding."
          " Without minority signal the model will collapse to predicting neutral.")
else:
    print(f"\nCommitted: {committed_pct:.1%} — sufficient for training.")

llm_df["label_id"] = llm_df["llm_buyin"].map(LABEL2ID)

In [ ]:
# ── Load human test set (held-out — NEVER trained on) ──────────────────────
test_raw = pd.read_csv(TEST_PATH)

assert "human_label" in test_raw.columns, (
    f"commitment_testset_queue.csv has no 'human_label' column for buyin axis. "
    f"Got: {list(test_raw.columns)}"
)

test_raw["human_label"] = test_raw["human_label"].astype(str).str.strip().str.lower()
valid_test = test_raw["human_label"].isin(LABEL2ID)
test_df = test_raw[valid_test].drop_duplicates("chunk_id").reset_index(drop=True)

assert len(test_df) >= 50, (
    f"Only {len(test_df)} labelled test rows — expected the completed 200-row test set."
)
print(f"Human test rows (buyin): {len(test_df)} / {len(test_raw)}")
print(f"\nHuman buyin distribution:")
print(test_df["human_label"].value_counts())

# ── Leakage guard: human test chunk_ids must NOT appear in training pool ────
leak = set(llm_df["chunk_id"]) & set(test_df["chunk_id"])
if leak:
    print(f"WARNING — {len(leak)} chunk_ids overlap between enrich and test set; "
          f"removing them from the TRAINING pool.")
    llm_df = llm_df[~llm_df["chunk_id"].isin(leak)].reset_index(drop=True)
assert not (set(llm_df["chunk_id"]) & set(test_df["chunk_id"])), "Leakage guard failed"
print("\nLeakage check passed — zero chunk_id overlap between train pool and human test set.")

In [ ]:
# ── Train/val split — SPLIT FIRST, then replicate ─────────────────────────
# Split BEFORE replication — never after (replicate-then-split leaks duplicates into val)
# Human rows → val ONLY. LLM rows → train ONLY. No mixing.
train_df, val_df = train_test_split(
    llm_df,
    test_size=VAL_FRAC,
    random_state=SEED,
    stratify=llm_df["label_id"],
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Before replication:")
print(f"  Train: {len(train_df):,}  Val: {len(val_df):,}")
print(f"  Train dist: {train_df['llm_buyin'].value_counts().to_dict()}")

# ── Replication: committed × 6, uncommitted × 4, neutral × 1 ─────────────
# 10× replication for both — targeting 70% recall on minority classes.
# Split MUST precede this step.
REPLICATE_COMMITTED   = 10
REPLICATE_UNCOMMITTED = 10

train_committed   = train_df[train_df["llm_buyin"] == "committed"]
train_uncommitted = train_df[train_df["llm_buyin"] == "uncommitted"]
train_neutral     = train_df[train_df["llm_buyin"] == "neutral"]

replicated_parts = [train_neutral]  # neutral × 1
for _ in range(REPLICATE_COMMITTED):
    replicated_parts.append(train_committed)
for _ in range(REPLICATE_UNCOMMITTED):
    replicated_parts.append(train_uncommitted)

train_df = pd.concat(replicated_parts, ignore_index=True).sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

print(f"\nAfter replication (committed × {REPLICATE_COMMITTED}, uncommitted × {REPLICATE_UNCOMMITTED}, neutral × 1):")
print(f"  Train: {len(train_df):,}  Val: {len(val_df):,}")
print(f"  Train dist: {train_df['llm_buyin'].value_counts().to_dict()}")
print(f"  Val dist:   {val_df['llm_buyin'].value_counts().to_dict()}")
print(f"  Test:  {len(test_df)} rows (human labels — final gate, untouched until the end)")

In [ ]:
# ── Tokenizer + dataset ───────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenising train ({len(train_df):,}) ...")
train_enc = tokenizer(train_df["text"].tolist(), truncation=True, max_length=MAX_LEN, padding=False)
print(f"Tokenising val   ({len(val_df):,}) ...")
val_enc   = tokenizer(val_df["text"].tolist(),   truncation=True, max_length=MAX_LEN, padding=False)
print("Done.")


class CommitmentDataset(Dataset):
    def __init__(self, encodings, label_ids):
        self.encodings = encodings
        self.labels    = label_ids

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = CommitmentDataset(train_enc, train_df["label_id"].tolist())
val_dataset   = CommitmentDataset(val_enc,   val_df["label_id"].tolist())
print(f"Train dataset : {len(train_dataset):,}")
print(f"Val dataset   : {len(val_dataset):,}")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")
print(f"Labels: {LABEL2ID}")

In [ ]:
# ── Metrics + Trainer ─────────────────────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    kappa = cohen_kappa_score(labels, preds)
    acc   = accuracy_score(labels, preds)
    return {"kappa": kappa, "accuracy": acc}


training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.06,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 1,          # BERT-large ckpt ~1.3 GB; Kaggle disk 20 GB
    load_best_model_at_end      = True,
    metric_for_best_model       = "kappa",
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    logging_steps               = 50,
    report_to                   = "none",
    seed                        = SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Version-aware tokenizer kwarg: transformers>=4.46 → processing_class=, else tokenizer=
import transformers
_trainer_kwargs = dict(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)
tv = tuple(int(x) for x in transformers.__version__.split(".")[:2])
if tv >= (4, 46):
    _trainer_kwargs["processing_class"] = tokenizer
else:
    _trainer_kwargs["tokenizer"] = tokenizer

# Plain Trainer — no class weights (replication handles balance at data level)
trainer = Trainer(**_trainer_kwargs)

steps_per_epoch = max(1, len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM))
print("Trainer configured (no class weights — replication corrects ~88% neutral bias).")
print(f"Best model selected by: kappa (val — LLM-labelled rows)")
print(f"~{steps_per_epoch} optimizer steps/epoch x {EPOCHS} epochs max")

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
print("Starting training ...")
train_result = trainer.train()
print("\nTraining complete.")
print(f"  Total steps   : {train_result.global_step}")
print(f"  Training loss : {train_result.training_loss:.4f}")

metrics = trainer.evaluate()
print(f"\nBest val (LLM-label) metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# ── Human test-set evaluation — the Stage 5b gate (buyin axis) ────────────
test_enc = tokenizer(
    test_df["text"].astype(str).tolist(),
    truncation=True, max_length=MAX_LEN, padding=False,
)
test_label_ids = test_df["human_label"].map(LABEL2ID).tolist()
test_dataset   = CommitmentDataset(test_enc, test_label_ids)

test_out   = trainer.predict(test_dataset)
logits_np  = test_out.predictions
probs      = torch.softmax(torch.tensor(logits_np, dtype=torch.float32), dim=-1).numpy()
preds      = np.argmax(logits_np, axis=-1)
preds_named = [ID2LABEL[p] for p in preds]
true_named  = test_df["human_label"].tolist()

acc   = accuracy_score(test_label_ids, preds)
kappa = cohen_kappa_score(test_label_ids, preds)

print("=" * 72)
print("  Human Test-Set Evaluation — SingBERT buyin distill v1")
print("=" * 72)
print(f"  Rows evaluated : {len(test_label_ids)}")
print(f"  Accuracy       : {acc:.3f}  ({acc*100:.1f}%)")
print(f"  Cohen's Kappa  : {kappa:.3f}")
print()
print(classification_report(true_named, preds_named, labels=LABEL_ORDER, digits=3, zero_division=0))
print("  Confusion matrix (rows=human, cols=predicted):")
cm = confusion_matrix(true_named, preds_named, labels=LABEL_ORDER)
print(pd.DataFrame(cm, index=[f"h_{c}" for c in LABEL_ORDER],
                       columns=[f"p_{c}" for c in LABEL_ORDER]))

# ── Per-class recall for gate check ───────────────────────────────────────
p_vals, r_vals, f_vals, _ = precision_recall_fscore_support(
    true_named, preds_named, labels=LABEL_ORDER, average=None, zero_division=0
)
recall_committed   = r_vals[LABEL_ORDER.index("committed")]
recall_uncommitted = r_vals[LABEL_ORDER.index("uncommitted")]

# ── Gate check ─────────────────────────────────────────────────────────────
GATE_KAPPA            = 0.50
GATE_UNCOMMITTED_REC  = 0.50
GATE_COMMITTED_REC    = 0.55

kappa_pass     = kappa              >= GATE_KAPPA
uncommit_pass  = recall_uncommitted >= GATE_UNCOMMITTED_REC
commit_pass    = recall_committed   >= GATE_COMMITTED_REC
gate_pass      = kappa_pass and uncommit_pass and commit_pass

print(f"\n  --- GATE ---")
print(f"  kappa >= {GATE_KAPPA}           : {kappa:.3f}   {'PASS' if kappa_pass else 'FAIL'}")
print(f"  uncommitted recall >= {GATE_UNCOMMITTED_REC}  : {recall_uncommitted:.3f}   {'PASS' if uncommit_pass else 'FAIL'}")
print(f"  committed recall >= {GATE_COMMITTED_REC}    : {recall_committed:.3f}   {'PASS' if commit_pass else 'FAIL'}")
print(f"  Overall gate: {'PASS — proceed to Stage 6' if gate_pass else 'FAIL — apply lexicon override + threshold tuning before proceeding'}")
print("=" * 72)

In [ ]:
# ── Save test predictions + summary ────────────────────────────────────────
eval_df = test_df.copy()
eval_df["singbert_buyin"]         = preds_named
eval_df["prob_buyin_committed"]    = probs[:, LABEL2ID["committed"]]
eval_df["prob_buyin_uncommitted"]  = probs[:, LABEL2ID["uncommitted"]]
eval_df["prob_buyin_neutral"]      = probs[:, LABEL2ID["neutral"]]
eval_df["buyin_correct"]           = eval_df["human_label"] == eval_df["singbert_buyin"]

eval_path = OUTPUT_DIR / "buyin_testset_eval.csv"
eval_df.to_csv(eval_path, index=False)
print(f"Saved -> {eval_path}")

summary = {
    "model"         : MODEL_NAME,
    "axis"          : "buyin",
    "architecture"  : "single-head AutoModelForSequenceClassification (3-class)",
    "label2id"      : LABEL2ID,
    "train_rows"    : len(train_dataset),
    "val_rows"      : len(val_dataset),
    "test_rows"     : len(test_label_ids),
    "accuracy"      : round(acc,   4),
    "kappa"         : round(kappa, 4),
    "uncommitted_recall": round(float(recall_uncommitted), 4),
    "committed_recall"  : round(float(recall_committed),   4),
    "gate_pass"     : bool(gate_pass),
    "max_len"       : MAX_LEN,
    "lr"            : LR,
    "epochs_max"    : EPOCHS,
    "replication"   : f"committed × {REPLICATE_COMMITTED}, uncommitted × {REPLICATE_UNCOMMITTED}, neutral × 1",
}
with open(OUTPUT_DIR / "eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# ── Save model + tokenizer + id2label ─────────────────────────────────────
import shutil

model_out = OUTPUT_DIR / "best_model"
trainer.save_model(str(model_out))
tokenizer.save_pretrained(str(model_out))
print(f"Model saved -> {model_out}")

# Save id2label for the infer notebook to validate against
id2label_path = model_out / "id2label.json"
with open(id2label_path, "w") as f:
    json.dump({"axis": "buyin", "id2label": ID2LABEL, "label2id": LABEL2ID}, f, indent=2)
print(f"Saved id2label -> {id2label_path}")

for fp in sorted(model_out.iterdir()):
    print(f"  {fp.name:<40} {fp.stat().st_size / 1e6:>8.1f} MB")

zip_path = str(OUTPUT_DIR.parent / "singbert_ns_buyin")
shutil.make_archive(zip_path, "zip", str(model_out))
zip_size = Path(zip_path + ".zip").stat().st_size / 1e6
print(f"\nZipped -> {zip_path}.zip  ({zip_size:.0f} MB)")
print()
print("Next steps:")
print("  1. Check buyin gate thresholds above (kappa ≥ 0.45, uncommitted recall ≥ 0.40, committed recall ≥ 0.30)")
print("  2. Upload singbert_ns_buyin.zip as a Kaggle dataset")
print("  3. Also run kaggle_distill_stance_v1.ipynb for the stance axis")
print("  4. Then run kaggle_infer_commitment_v1.ipynb for full 737k dual-axis inference")

## Gate thresholds (v5 — updated for enriched training data)

Pass criteria before proceeding to inference:
- **kappa ≥ 0.50**
- **uncommitted recall ≥ 0.50** — model alone target; lexicon override + threshold tuning push to 0.70
- **committed recall ≥ 0.55**

If gate fails, apply lexicon override and threshold tuning (see roadmap) before re-evaluating.

Expected ranges after enriched retrain:
- Uncommitted recall: 0.55–0.65 (model alone) → 0.65–0.72 (+ lexicon + threshold)
- Committed recall:   0.60–0.70